dataset without battery

In [ ]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario — NO BESS
# - runs in chunks
# - different seed per chunk
# - different output folder per chunk
# - PE computed from static edges each chunk (node_pe_k > 0, no CSV)
# ============================================================
import os
import math
import time
from pathlib import Path

# ---------------- user controls ----------------
INCLUDE_BESS = False

TOTAL_SCENARIOS = 2000
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20320230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

# Only used if INCLUDE_BESS is True
BESS_TOTAL_MVA_MEAN = 0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_CANDIDATE_NODES_150 = []  # fill if you set INCLUDE_BESS = True

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

CHUNK_ROOT = Path(r"H:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    os.getcwd(),
]

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

# ---------------- preflight: chunk parent template (optional) ----------------
# Probes first chunk folder name pattern — fails early if chunk_parent layout wrong
root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS}")

for chunk_idx in range(n_chunks):
    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
    )

    if INCLUDE_BESS:
        if not BESS_CANDIDATE_NODES_150:
            raise ValueError("INCLUDE_BESS=True requires a non-empty BESS_CANDIDATE_NODES_150 list.")
        gen_kw.update(
            bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
            bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
            bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
            bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
            bess_q_frac_max=float(BESS_Q_FRAC_MAX),
            bess_candidate_nodes_override=BESS_CANDIDATE_NODES_150,
        )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    print("Saved:")
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    print(" -", ns["NODE_CSV"])

print("\nAll chunks finished.")

datset with battery

In [1]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario -- WITH BESS
# - runs in chunks
# - different seed per chunk
# - different output folder per chunk
# - auto-discovers eligible 3-phase MV BESS buses
# - writes both explicit-BESS mvagg and compatibility mvagg
#   where BESS P/Q is folded into load P/Q
# ============================================================
import os
import math
import time
import re
import shutil
from pathlib import Path

import pandas as pd

# ---------------- user controls ----------------
INCLUDE_BESS = True

TOTAL_SCENARIOS = 200
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20420230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

BESS_TOTAL_MVA_MEAN = 4.0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_SCATTERED_BUS_TARGET = 72
BESS_MIN_ELECTRICAL_DISTANCE_OHM = 0.5

# Disk-saving options (important on K: / Google Drive)
SAVE_EXPLICIT_BESS_MVAGG = False   # training only needs 3-feature mvagg.csv
DELETE_RAW_NODE_CSV_AFTER_MVAGG = True
MIN_FREE_GB_BEFORE_CHUNK = 2.0

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    os.getcwd(),
]

# This folder is separate from the no-BESS dataset.
CHUNK_ROOT = Path(r"K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40")
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
print("CHUNK_ROOT:", CHUNK_ROOT)

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

FEEDER_DIR = Path(script_path).resolve().parent / "8500 nodes with solar unbalanced"
TX_DSS = FEEDER_DIR / "LoadXfmrs.dss"
if not TX_DSS.exists():
    raise FileNotFoundError(f"Missing file: {TX_DSS}")


def tok(s: str) -> str:
    return str(s).strip().lower()


def bus_base(node_or_bus: str) -> str:
    return tok(node_or_bus).split(".")[0]


def build_tx_map() -> pd.DataFrame:
    wdg_bus_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
    map_rows = []

    for line in TX_DSS.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s.lower().startswith("new transformer."):
            continue

        w = {int(k): tok(v) for k, v in wdg_bus_re.findall(s)}
        if not (1 in w and 2 in w and 3 in w):
            continue

        x2 = bus_base(w[2])
        x3 = bus_base(w[3])
        if not (x2.startswith("x") and x3.startswith("x")):
            continue

        mv_node = w[1]
        sx_set = {"s" + x2, "s" + x3}
        for sx_bus in sx_set:
            map_rows.append((mv_node, sx_bus))

    out = pd.DataFrame(map_rows, columns=["mv_node", "sx_bus"]).drop_duplicates()
    dup = out.duplicated(["mv_node", "sx_bus"]).sum()
    assert dup == 0, f"Unexpected duplicate mappings: {dup}"
    return out


def build_scattered_bess_candidate_csv(out_dir: Path) -> tuple[list[str], Path]:
    ns["_compile_8500_unbalanced_daily_setup"]()
    ns["_detach_daily_loadshape_from_loads"]()
    node_names_all, _, _, _ = ns["inj"].get_all_bus_phase_nodes()
    node_names_graph = ns["_filter_out_x_sx_nodes"](node_names_all)
    node_set_all = {str(n).strip().lower() for n in node_names_all}

    # Preferred path: transformer-adjacent MV load buses.
    auto_nodes = ns["_collect_bess_candidate_nodes_from_mv_load_transformers"](node_names_all)
    bus_to_nodes = ns["_candidate_three_phase_buses_from_nodes"](auto_nodes)
    method = "transformer-adjacent MV load buses"

    # Fallback: any valid 3-phase MV bus in the retained feeder graph.
    if not bus_to_nodes:
        fallback_nodes = []
        for node in node_names_all:
            s = str(node).strip().lower()
            if "." not in s:
                continue
            bus, phs = s.rsplit(".", 1)
            if bus.startswith(("x", "sx", "sourcebus", "hvmv_sub_hsb", "_hvmv_sub")):
                continue
            try:
                ph = int(phs)
            except Exception:
                continue
            if ph not in (1, 2, 3):
                continue
            try:
                ns["dss"].Circuit.SetActiveBus(bus)
                kvb = float(ns["dss"].Bus.kVBase())
            except Exception:
                kvb = float("nan")
            if pd.notna(kvb) and (kvb > 1.0) and (kvb <= 40.0):
                fallback_nodes.append(f"{bus}.{ph}")
        bus_to_nodes = ns["_candidate_three_phase_buses_from_nodes"](fallback_nodes)
        method = "fallback all 3-phase MV buses"

    if not bus_to_nodes:
        raise RuntimeError("No eligible 3-phase MV BESS candidate buses were found, even after MV fallback.")

    # Build electrical-distance metadata so the saved CSV is spread over the feeder.
    edge_tmp = out_dir / "_tmp_candidate_edges.csv"
    ns["inj"].extract_static_phase_edges_to_csv(
        node_names_master=node_names_graph,
        edge_csv_path=str(edge_tmp),
        excluded_buses=(),
    )
    node_to_dist = ns["lt_dist"]._compute_electrical_distance_from_source(node_names_graph, str(edge_tmp))

    rows = []
    dropped_regxfmr = []
    dropped_near_source = []
    for bus, nodes in sorted(bus_to_nodes.items()):
        dist_vals = [float(node_to_dist.get(node, float("nan"))) for node in nodes]
        finite = [x for x in dist_vals if pd.notna(x)]
        dist = float(sum(finite) / len(finite)) if finite else float("nan")

        if str(bus).lower().startswith("regxfmr_"):
            dropped_regxfmr.append((bus, dist))
            continue
        if pd.notna(dist) and float(dist) < float(BESS_MIN_ELECTRICAL_DISTANCE_OHM):
            dropped_near_source.append((bus, dist))
            continue

        rows.append(
            {
                "bus": bus,
                "node_1": nodes[0],
                "node_2": nodes[1],
                "node_3": nodes[2],
                "electrical_distance_ohm": dist,
            }
        )

    cand_df = pd.DataFrame(rows).sort_values(["electrical_distance_ohm", "bus"], kind="stable").reset_index(drop=True)
    if cand_df.empty:
        raise RuntimeError(
            "Candidate filtering removed all buses. "
            "Reduce BESS_MIN_ELECTRICAL_DISTANCE_OHM or relax exclusions."
        )
    n_total = len(cand_df)
    target = max(int(BESS_NUM_NODES_MAX), min(int(BESS_SCATTERED_BUS_TARGET), n_total))

    if n_total > target:
        picks = sorted({int(round(i * (n_total - 1) / (target - 1))) for i in range(target)})
        while len(picks) < target:
            for idx in range(n_total):
                if idx not in picks:
                    picks.append(idx)
                if len(picks) == target:
                    break
        cand_df = cand_df.iloc[sorted(picks)].reset_index(drop=True)

    cand_df.insert(0, "selection_rank", range(1, len(cand_df) + 1))
    cand_df.insert(1, "selection_method", method)

    csv_path = out_dir / "bess_candidate_buses_scattered_3ph_mv.csv"
    cand_df.to_csv(csv_path, index=False)

    candidate_nodes = []
    for _, row in cand_df.iterrows():
        candidate_nodes.extend([str(row["node_1"]), str(row["node_2"]), str(row["node_3"])])

    # ---------------- sanity checks ----------------
    req_cols = {
        "selection_rank", "selection_method", "bus", "node_1", "node_2", "node_3", "electrical_distance_ohm"
    }
    miss = req_cols - set(cand_df.columns)
    if miss:
        raise RuntimeError(f"Candidate CSV missing required columns: {sorted(miss)}")

    bus_unique = int(cand_df["bus"].nunique())
    if bus_unique != len(cand_df):
        raise RuntimeError("Candidate CSV contains duplicate buses.")

    node_cols = ["node_1", "node_2", "node_3"]
    phase_ok = True
    missing_nodes = []
    bad_phase_rows = []
    for _, row in cand_df.iterrows():
        nodes = [str(row[c]).strip().lower() for c in node_cols]
        buses = {n.rsplit(".", 1)[0] for n in nodes}
        phases = sorted(int(n.rsplit(".", 1)[1]) for n in nodes)
        if len(buses) != 1 or phases != [1, 2, 3]:
            phase_ok = False
            bad_phase_rows.append((row["bus"], nodes))
        for n in nodes:
            if n not in node_set_all:
                missing_nodes.append(n)

    if not phase_ok:
        raise RuntimeError(f"Candidate CSV has non-3phase rows, examples: {bad_phase_rows[:3]}")
    if missing_nodes:
        raise RuntimeError(f"Candidate CSV contains nodes missing from circuit, examples: {missing_nodes[:6]}")

    dist_series = cand_df["electrical_distance_ohm"].astype(float)
    dist_finite = dist_series[pd.notna(dist_series)]
    dist_min = float(dist_finite.min()) if len(dist_finite) else float("nan")
    dist_max = float(dist_finite.max()) if len(dist_finite) else float("nan")
    dist_q = dist_finite.quantile([0.1, 0.5, 0.9]).to_dict() if len(dist_finite) else {}
    gaps = dist_finite.sort_values().diff().dropna()
    gap_min = float(gaps.min()) if len(gaps) else float("nan")
    gap_med = float(gaps.median()) if len(gaps) else float("nan")

    print(
        f"Saved {len(cand_df)} scattered candidate 3-phase MV buses "
        f"({len(candidate_nodes)} phase nodes) using {method}."
    )
    print("Candidate CSV:", csv_path)
    print("First few candidate buses:", cand_df["bus"].head(10).tolist())
    print(
        f"Filtered out regxfmr_* buses: {len(dropped_regxfmr)} | "
        f"near-source buses (< {float(BESS_MIN_ELECTRICAL_DISTANCE_OHM):.3f} ohm): {len(dropped_near_source)}"
    )
    if dropped_regxfmr:
        print(" - sample regxfmr drops:", dropped_regxfmr[:5])
    if dropped_near_source:
        print(" - sample near-source drops:", dropped_near_source[:5])
    print("Sanity checks:")
    print(f" - unique buses: {bus_unique}")
    print(f" - phase rows valid [.1,.2,.3 on same bus]: {phase_ok}")
    print(f" - nodes present in circuit: {len(missing_nodes) == 0}")
    print(f" - electrical distance finite rows: {int(len(dist_finite))}/{len(cand_df)}")
    print(f" - distance range ohm: [{dist_min:.3f}, {dist_max:.3f}]")
    if dist_q:
        print(
            " - distance quantiles ohm: "
            f"q10={float(dist_q.get(0.1, float('nan'))):.3f}, "
            f"q50={float(dist_q.get(0.5, float('nan'))):.3f}, "
            f"q90={float(dist_q.get(0.9, float('nan'))):.3f}"
        )
    print(f" - spacing gaps ohm: min={gap_min:.3f}, median={gap_med:.3f}")
    return candidate_nodes, csv_path


def build_mvagg_outputs_for_chunk(chunk_dir: Path, map_long: pd.DataFrame) -> dict:
    in_csv = chunk_dir / "gnn_node_features_and_targets.csv"
    idx_csv = chunk_dir / "gnn_node_index_master.csv"
    out_explicit = chunk_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv"
    out_compat = chunk_dir / "gnn_node_features_and_targets_mvagg.csv"

    if not in_csv.exists() or not idx_csv.exists():
        raise FileNotFoundError(f"Missing inputs for mvagg build under: {chunk_dir}")

    df = pd.read_csv(in_csv)
    idx = pd.read_csv(idx_csv, usecols=["node"])
    allowed_nodes = set(idx["node"].astype(str).str.strip().str.lower())

    df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
    df["bus"] = df["node_lc"].str.split(".").str[0]

    sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].copy()
    sx = sx.rename(columns={"bus": "sx_bus"})

    sx_join = sx.merge(map_long, on="sx_bus", how="inner")
    mv_agg = (
        sx_join.groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
        .sum()
        .rename(columns={"p_load_kw": "p_load_kw_agg", "q_load_kvar": "q_load_kvar_agg"})
    )

    df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
    df2 = df2[df2["node_lc"].isin(allowed_nodes)].copy()

    # Replace raw LV-side loads with MV-aggregated equivalent load.
    df2["p_load_kw"] = 0.0
    df2["q_load_kvar"] = 0.0
    df2 = df2.merge(
        mv_agg,
        left_on=["sample_id", "node_lc"],
        right_on=["sample_id", "mv_node"],
        how="left",
    )

    hit = df2["p_load_kw_agg"].notna()
    df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_load_kw_agg"]
    df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_load_kvar_agg"]

    for c in ["node_lc", "bus", "mv_node", "p_load_kw_agg", "q_load_kvar_agg"]:
        if c in df2.columns:
            df2.drop(columns=c, inplace=True)

    # Save only what we need for training augmentation.
    if SAVE_EXPLICIT_BESS_MVAGG:
        df2.to_csv(out_explicit, index=False)

    compat = df2.copy()
    compat["p_load_kw"] = compat["p_load_kw"].astype(float) + compat["p_bess_kw"].fillna(0.0).astype(float)
    compat["q_load_kvar"] = compat["q_load_kvar"].astype(float) + compat["q_bess_kvar"].fillna(0.0).astype(float)
    compat.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")
    compat.to_csv(out_compat, index=False)

    if DELETE_RAW_NODE_CSV_AFTER_MVAGG and in_csv.exists():
        in_csv.unlink()
        print("   deleted raw:", in_csv.name)

    n_samples = int(df2["sample_id"].nunique()) if len(df2) else 0
    rows_per_sample = df2.groupby("sample_id").size() if len(df2) else pd.Series(dtype=int)
    rmin = int(rows_per_sample.min()) if len(rows_per_sample) else 0
    rmax = int(rows_per_sample.max()) if len(rows_per_sample) else 0
    print("Post-processed:")
    if SAVE_EXPLICIT_BESS_MVAGG:
        print(" -", out_explicit)
    print(" -", out_compat)
    print(f"   rows={len(df2)} samples={n_samples} rows_per_sample=[{rmin}, {rmax}]")
    return {
        "rows": int(len(df2)),
        "samples": n_samples,
        "rows_per_sample_min": rmin,
        "rows_per_sample_max": rmax,
    }


BESS_CANDIDATE_NODES_AUTO, BESS_CANDIDATE_CSV = build_scattered_bess_candidate_csv(CHUNK_ROOT)
TX_MAP_LONG = build_tx_map()

root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS}")

free_gb = shutil.disk_usage(CHUNK_ROOT).free / (1024 ** 3)
print(f"Free space on drive: {free_gb:.2f} GB")
if free_gb < MIN_FREE_GB_BEFORE_CHUNK:
    raise OSError(
        f"Not enough free space on {CHUNK_ROOT.drive}: {free_gb:.2f} GB free, "
        f"need at least {MIN_FREE_GB_BEFORE_CHUNK:.2f} GB"
    )

for chunk_idx in range(n_chunks):
    free_gb = shutil.disk_usage(CHUNK_ROOT).free / (1024 ** 3)
    if free_gb < MIN_FREE_GB_BEFORE_CHUNK:
        raise OSError(
            f"Stopping before chunk {chunk_idx + 1}: only {free_gb:.2f} GB free on "
            f"{CHUNK_ROOT.drive} (need {MIN_FREE_GB_BEFORE_CHUNK:.2f} GB)"
        )

    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
        bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
        bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
        bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
        bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
        bess_q_frac_max=float(BESS_Q_FRAC_MAX),
        bess_candidate_nodes_override=BESS_CANDIDATE_NODES_AUTO,
    )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)
    mvagg_stats = build_mvagg_outputs_for_chunk(out_dir, TX_MAP_LONG)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    print(
        f"MVAGG summary | rows={mvagg_stats['rows']} | samples={mvagg_stats['samples']} "
        f"| rows/sample=[{mvagg_stats['rows_per_sample_min']}, {mvagg_stats['rows_per_sample_max']}]"
    )
    print("Saved:")
    print(" -", BESS_CANDIDATE_CSV)
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    if not DELETE_RAW_NODE_CSV_AFTER_MVAGG:
        print(" -", ns["NODE_CSV"])
    print(" -", out_dir / "gnn_node_features_and_targets_mvagg.csv")
    if SAVE_EXPLICIT_BESS_MVAGG:
        print(" -", out_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv")

print("\nAll chunks finished.")

CHUNK_ROOT: K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40
CWD: c:\Users\alita\OneDrive\Desktop\GNN2
SCRIPT: C:\Users\alita\OneDrive\Desktop\GNN2\run_original_style_dataset_8500_unbalanced.py
[saved] phase-edge CSV -> K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40\_tmp_candidate_edges.csv | rows=7628 | cols=17 | bidirectional=True
Saved 72 scattered candidate 3-phase MV buses (216 phase nodes) using fallback all 3-phase MV buses.
Candidate CSV: K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40\bess_candidate_buses_scattered_3ph_mv.csv
First few candidate buses: ['m1209797', 'm3032980', 'm1209774', 'l3139366', 'e184626', 'm1149235', 'l3081380', 'm1142839', 'l2674047', 'm1142818']
Filtered out regxfmr_* buses: 4 | near-source buses (< 0.500 ohm): 15
 - sample regxfmr drops: [('regxfmr_190-7361', 6.553390629007642), ('regxfmr_190-8581', 4.481039542293969), ('regxfmr_190-8593', 6.972486484840

post processing

In [ ]:
import re
from pathlib import Path
import pandas as pd

# ---------------- paths ----------------
FEEDER_DIR = Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500 nodes with solar unbalanced")
CHUNK_PARENT = Path(r"K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40")
TX_DSS = FEEDER_DIR / "LoadXfmrs.dss"

if not TX_DSS.exists():
    raise FileNotFoundError(f"Missing file: {TX_DSS}")
if not CHUNK_PARENT.exists():
    raise FileNotFoundError(f"Missing folder: {CHUNK_PARENT}")

def tok(s: str) -> str:
    return str(s).strip().lower()

def bus_base(node_or_bus: str) -> str:
    return tok(node_or_bus).split(".")[0]

# ---------------- 1) build global transformer map once ----------------
wdg_bus_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
map_rows = []

for line in TX_DSS.read_text(encoding="utf-8", errors="ignore").splitlines():
    s = line.strip()
    if not s.lower().startswith("new transformer."):
        continue

    w = {int(k): tok(v) for k, v in wdg_bus_re.findall(s)}
    if not (1 in w and 2 in w and 3 in w):
        continue

    x2 = bus_base(w[2])
    x3 = bus_base(w[3])
    if not (x2.startswith("x") and x3.startswith("x")):
        continue

    mv_node = w[1]                   # node-level MV target (keep phase)
    sx_set = {"s" + x2, "s" + x3}    # dedupe prevents double counting
    for sx_bus in sx_set:
        map_rows.append((mv_node, sx_bus))

map_long = pd.DataFrame(map_rows, columns=["mv_node", "sx_bus"]).drop_duplicates()

dup = map_long.duplicated(["mv_node", "sx_bus"]).sum()
assert dup == 0, f"Unexpected duplicate mappings: {dup}"

print("Global mapping ready.")
print("Unique MV nodes mapped:", map_long["mv_node"].nunique())
print("Mapping rows (mv_node,sx_bus):", len(map_long))

# ---------------- 2) process all chunks ----------------
chunk_dirs = sorted([p for p in CHUNK_PARENT.iterdir() if p.is_dir() and p.name.startswith("run_")])
print(f"Found {len(chunk_dirs)} chunk folders.")

summary = []
failed = []
skipped = []

for i, chunk_dir in enumerate(chunk_dirs, 1):
    in_csv = chunk_dir / "gnn_node_features_and_targets.csv"
    idx_csv = chunk_dir / "gnn_node_index_master.csv"
    out_explicit = chunk_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv"
    out_compat = chunk_dir / "gnn_node_features_and_targets_mvagg.csv"

    try:
        if not in_csv.exists() or not idx_csv.exists():
            raise FileNotFoundError(f"Missing input(s): {in_csv.exists()=}, {idx_csv.exists()=}")

        if out_explicit.exists() and out_explicit.stat().st_size > 0 and out_compat.exists() and out_compat.stat().st_size > 0:
            skipped.append(chunk_dir.name)
            print(f"[{i}/{len(chunk_dirs)}] SKIP {chunk_dir.name} (already post-processed)")
            continue

        df = pd.read_csv(in_csv)
        need = {"sample_id", "node", "p_load_kw", "q_load_kvar", "p_pv_kw", "p_bess_kw", "q_bess_kvar"}
        miss = need - set(df.columns)
        if miss:
            raise ValueError(f"Missing required columns: {miss}")

        idx = pd.read_csv(idx_csv, usecols=["node"])
        allowed_nodes = set(idx["node"].astype(str).str.strip().str.lower())

        df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
        df["bus"] = df["node_lc"].str.split(".").str[0]

        # aggregate SX -> exact MV node for load channels only
        sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].copy()
        sx = sx.rename(columns={"bus": "sx_bus"})

        sx_join = sx.merge(map_long, on="sx_bus", how="inner")
        mv_agg = (
            sx_join.groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
            .sum()
            .rename(columns={"p_load_kw": "p_load_kw_agg", "q_load_kvar": "q_load_kvar_agg"})
        )

        # remove X/SX and filter to index master
        df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
        df2 = df2[df2["node_lc"].isin(allowed_nodes)].copy()

        # strict MV load assignment from aggregated SX load
        df2["p_load_kw"] = 0.0
        df2["q_load_kvar"] = 0.0

        df2 = df2.merge(
            mv_agg,
            left_on=["sample_id", "node_lc"],
            right_on=["sample_id", "mv_node"],
            how="left"
        )

        hit = df2["p_load_kw_agg"].notna()
        df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_load_kw_agg"]
        df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_load_kvar_agg"]

        for c in ["node_lc", "bus", "mv_node", "p_load_kw_agg", "q_load_kvar_agg"]:
            if c in df2.columns:
                df2.drop(columns=c, inplace=True)

        # explicit-BESS mvagg output: keeps 5 dynamic features
        df2.to_csv(out_explicit, index=False)

        # compatibility mvagg output: folds BESS into load, drops BESS columns
        compat = df2.copy()
        compat["p_load_kw"] = compat["p_load_kw"].astype(float) + compat["p_bess_kw"].fillna(0.0).astype(float)
        compat["q_load_kvar"] = compat["q_load_kvar"].astype(float) + compat["q_bess_kvar"].fillna(0.0).astype(float)
        compat.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")
        compat.to_csv(out_compat, index=False)

        # quick stats
        n_samples = int(df2["sample_id"].nunique()) if len(df2) else 0
        rows_per_sample = df2.groupby("sample_id").size() if len(df2) else pd.Series(dtype=int)
        rmin = int(rows_per_sample.min()) if len(rows_per_sample) else 0
        rmax = int(rows_per_sample.max()) if len(rows_per_sample) else 0

        summary.append({
            "chunk": chunk_dir.name,
            "samples": n_samples,
            "rows": int(len(df2)),
            "rows_per_sample_min": rmin,
            "rows_per_sample_max": rmax,
            "saved_explicit": str(out_explicit),
            "saved_compat": str(out_compat),
        })

        print(
            f"[{i}/{len(chunk_dirs)}] OK  {chunk_dir.name} "
            f"-> rows={len(df2)} samples={n_samples} "
            f"rps=[{rmin},{rmax}]"
        )
        print("   explicit:", out_explicit.name)
        print("   compat  :", out_compat.name)

    except Exception as e:
        failed.append((chunk_dir.name, str(e)))
        print(f"[{i}/{len(chunk_dirs)}] FAIL {chunk_dir.name}: {e}")

print("\nDone.")
print(f"Success: {len(summary)} | Skipped: {len(skipped)} | Failed: {len(failed)}")

if skipped:
    print("\nSkipped (already post-processed):")
    for name in skipped:
        print(f"- {name}")

if failed:
    print("\nFailures:")
    for name, err in failed:
        print(f"- {name}: {err}")

inspection

In [ ]:
import pandas as pd
from pathlib import Path

in_csv = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked/run_001_scen_0000_0049_seed_20360133/gnn_node_features_and_targets_mvagg.csv")
out_csv = in_csv.with_name("gnn_node_features_and_targets_mvagg_sample0.csv")

df = pd.read_csv(in_csv)

# first sample by appearance order in file
first_sid = df["sample_id"].iloc[0]
df_first = df[df["sample_id"] == first_sid].copy()

df_first.to_csv(out_csv, index=False)
print("first sample_id:", first_sid)
print("rows saved:", len(df_first))
print("saved:", out_csv)

plotting

In [ ]:
%matplotlib inline
import os
import sys
from pathlib import Path

# ── find repo (Colab clone, GNN2_REPO_ROOT, Windows path, or cwd) ─────────────
def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "nonunique_notebook_bootstrap.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone to /content/GNN2 first. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )

REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Reload after `git pull` on Colab so you get p_pv_kw fix, wide warm-starts, discrete aux metrics
for _m in (
    "nonunique_notebook_bootstrap",
    "nonunique_da_gps_warmstart_band_daily",
    "nonunique_opendss_daily",
    "nonunique_daily_experiment",
    "run_da_gps_daily_opendss_compare",
    "compare_gnn_inference_utils",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook
from nonunique_opendss_daily import DailySimConfig
from nonunique_da_gps_warmstart_band_daily import run_da_gps_warmstart_band_daily

# ── checkpoint source ─────────────────────────────────────────────────────────
# Default: shipped CCE baseline (h=96, L=2, reg CE, 4 meta-aux heads)
USE_FINETUNED_CHECKPOINT = False
# After fine-tune cell 13, set True and point at the new Drive run folder:
FINETUNE_RUN_DIR = Path(
    "/content/drive/MyDrive/datasets_gnn2/runs/"
    "da_gps_finetune_withder_l2_h96_regce_<timestamp>"
)

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="auto",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
)

if USE_FINETUNED_CHECKPOINT:
    finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
    ckpt = finetune_run / "da_gps_multitask_best.pt"
    if not ckpt.is_file():
        ckpt = finetune_run / "training_last.pt"
    if not ckpt.is_file():
        raise FileNotFoundError(f"No checkpoint in {finetune_run}")
    boot = boot.__class__(
        **{
            **boot.__dict__,
            "run_dir": finetune_run,
            "checkpoint": ckpt,
        }
    )
    print(f"[checkpoint] fine-tuned: {ckpt}")

# ── knobs (same on Colab and local) ───────────────────────────────────────────
INCLUDE_DER      = True
DER_MAX_KW       = 500.0
DER_MAX_KVAR     = 50.0
DER_BUS          = "l2801895"

N_WARM_STARTS    = 12                 # more starts → wider cloud (slower)
WARM_START_MODE  = "wide"             # "uniform" | "corners" | "wide"
WARM_START_RANDOMIZE_STATIC_CAPS = False  # True = wilder cap bands
WARMSTART_SEED   = 42

STEP_MIN         = 5
DAILY_STRESS     = 0.0
SCENARIO_SCALE   = 1.0
REF_SAMPLE_INDEX = 0

PLOT_ALL_CACHE_NODES = True
PLOT_ALL_MAX_NODES   = 10              # 0 = all cache∩circuit nodes
PLOT_REG_CAP         = True
PLOT_META_AUX        = True
PLOT_WARMSTART_LINES = True
SHOW_INLINE          = False
VOLTAGE_PLOT_DPI     = 96
GNN_BATCH_STEPS      = None           # or 8; also env GNN_BATCH_STEPS

OUT_DIR = boot.out_dir                # or Path("/content/drive/MyDrive/.../warmstart_runs/my_tag")

# ── run ─────────────────────────────────────────────────────────────────────
cfg = DailySimConfig(
    step_min=STEP_MIN,
    include_der=INCLUDE_DER,
    der_nominal_kw=float(DER_MAX_KW if INCLUDE_DER else 0.0),
    der_nominal_kvar=float(DER_MAX_KVAR if INCLUDE_DER else 0.0),
    der_bus=str(DER_BUS),
    der_profile_csv=boot.der_profile,
    da_gps_run_dir=boot.run_dir,
    da_gps_cache_pt=boot.cache_pt,
    da_gps_checkpoint=boot.checkpoint,
)

print(
    f"env={'Colab' if boot.on_colab else 'local'}  device={boot.device}\n"
    f"checkpoint={boot.checkpoint}\n"
    f"DER={'ON' if INCLUDE_DER else 'OFF'}  "
    f"warm_starts={N_WARM_STARTS}  mode={WARM_START_MODE!r}  step_min={STEP_MIN}"
)

result = run_da_gps_warmstart_band_daily(
    cfg,
    n_warm_starts=N_WARM_STARTS,
    warm_start_mode=WARM_START_MODE,
    warm_start_randomize_static_caps=WARM_START_RANDOMIZE_STATIC_CAPS,
    seed=WARMSTART_SEED,
    load_profile_path=boot.load_profile,
    pv_profile_path=boot.irr_profile,
    ref_sample_index=REF_SAMPLE_INDEX,
    scenario_scale=SCENARIO_SCALE,
    daily_stress=DAILY_STRESS,
    plot_all_cache_nodes=PLOT_ALL_CACHE_NODES,
    plot_all_max_nodes=PLOT_ALL_MAX_NODES,
    out_dir=OUT_DIR,
    voltage_plot_dpi=VOLTAGE_PLOT_DPI,
    plot_reg_cap=PLOT_REG_CAP,
    plot_meta_aux=PLOT_META_AUX,
    plot_warmstart_lines=PLOT_WARMSTART_LINES,
    show=SHOW_INLINE,
    device=boot.device,
    gnn_batch_steps=GNN_BATCH_STEPS,
)

inside = result["da_gps_inside_band_frac"]
proximity = result["da_gps_cloud_proximity"]
v_inside = inside["voltage"]
v_prox = proximity["voltage"]
print("\nOutputs:", result["out_dir"])
print(f"Nodes: {len(result['collect_nodes'])}")
print(f"Mean inside-band |V|: {100.0 * sum(v_inside.values()) / max(1, len(v_inside)):.1f}%")
print(f"Mean cloud proximity |V|: {sum(v_prox.values()) / max(1, len(v_prox)):.3f}")
for group in ("regulator", "capacitor", "meta_aux"):
    items = inside.get(group) or {}
    prox_items = proximity.get(group) or {}
    if items:
        vals = [v for v in items.values() if v == v]
        if vals:
            print(f"Mean inside-band {group}: {100.0 * sum(vals) / len(vals):.1f}%")
    if prox_items:
        pvals = [v for v in prox_items.values() if v == v]
        if pvals:
            print(f"Mean cloud proximity {group}: {sum(pvals) / len(pvals):.3f}")
inside

[bootstrap] env=local  repo=C:\Users\alita\OneDrive\Desktop\GNN2
[bootstrap] device=cpu  cache=run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=C:\Users\alita\OneDrive\Desktop\GNN2\warmstart_band_runs\20260709_143047
env=local  device=cpu
checkpoint=C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE\training_last.pt
DER=ON  warm_starts=12  mode='wide'  step_min=5
DA-GPS warm-start band daily (OpenDSS snapshot + N random controller inits/step)
  step_min=5 min, npts=288, n_warm_starts=12
  warm_start_mode='wide'  randomize_static_caps=False
  load/PV profiles: C:\Users\alita\OneDrive\Desktop\GNN2\a representativ days\load_day_004.csv / C:\Users\alita\OneDrive\Desktop\GNN2\a representativ days\irr_day_004.csv
  cache .pt: C:\Users\alita\OneDrive\D

compare with this

In [ ]:
# --- Lighter DA-GPS training (long-running; run on Colab GPU or local CUDA) ---
import os
import sys
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
WIN_CHUNK_DEFAULT = Path(r"D:\datasets\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]

def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _smoke_chunk_subdir_glob(chunk_parent: Path, count: int) -> str:
    """First `count` run_* names, comma-separated for --chunk_subdir_glob."""
    names = [p.name for p in _sorted_run_chunk_dirs(chunk_parent)]
    if len(names) < count:
        raise ValueError(
            f"SMOKE_CHUNK_COUNT={count} but only {len(names)} run_* under {chunk_parent}"
        )
    return ",".join(names[:count])


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- smoke / full-run toggles (cells 6 baseline & 17 physics; cell 18 compares) ---
SMOKE_TEST = True
SMOKE_CHUNK_COUNT = 3  # first N run_* alphabetically; 3-5 = fast multi-chunk (not all 40)
SMOKE_EPOCHS = 15      # low for speed; more epochs = stabler MAE, ~linear time
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
_DA_CACHE_NAME = "da_gps_chunked_mvagg_smoke_gine" if SMOKE_TEST else "da_gps_chunked_mvagg_full_gine"

# --- paths: auto-detect Colab + Drive; override below if needed ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = MYDRIVE_DATA / "runs"  # persisted on Drive; /content/GNN-Sandia is ephemeral
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"

NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT

# Optional overrides (use POSIX paths on Colab, not D:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"

# --- lighter architecture (sweeps: LIGHTER_LAYERS=1, LIGHTER_HIDDEN=48) ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 4  # current shipped model used 4 at h=96

PHYSICS_WEIGHT = 0.0  # baseline; use 0.01 for physics-informed


# Explicit PF balance nodes when physics on (1177 hetero MV load nodes (node-name keyed; chunk-safe); skips auto mask refinement)
PF_BALANCE_NODE_LIST_CSV = REPO / "colab_pf_data/pf_balance_nodes_explicit.csv"

META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = 200
    PATIENCE = 30
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

# --- PF topology root (repo colab_pf_data/ after git pull, or Drive dailyagg fallback) ---
from gnn2_pf_data_paths import PF_CAP_NODES_REL, PF_REG_CATALOG_REL, resolve_pf_catalog_paths

PF_DATA_ROOT = None
if PHYSICS_WEIGHT > 0:
    _reg_cat, _cap_map, PF_DATA_ROOT = resolve_pf_catalog_paths(
        repo=REPO,
        preferred_root=None,
        chunk_parent=chunk_parent,
    )

print("=== Preflight ===")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"CHUNK_PARENT:   {chunk_parent}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")

if PHYSICS_WEIGHT > 0:
    assert PF_DATA_ROOT is not None
    _pf_checks = [
        ("reg_catalog", PF_DATA_ROOT / PF_REG_CATALOG_REL),
        ("cap_nodes", PF_DATA_ROOT / PF_CAP_NODES_REL),
        ("electrical_distance", PF_DATA_ROOT / "electrical_distance_from_substation.csv"),
        (
            "hetero_mv_nodes",
            PF_DATA_ROOT / "Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer.csv",
        ),
        ("bus_kv_cache", PF_DATA_ROOT / "bus_kv_base_by_node.csv"),
    ]
    print(f"PF_DATA_ROOT:   {PF_DATA_ROOT}")
    for label, p in _pf_checks:
        status = "OK" if p.is_file() else "MISSING"
        print(f"  PF {label}: {status}  {p}")
        if not p.is_file():
            raise FileNotFoundError(f"Physics preflight missing {label}: {p}")
else:
    print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")

print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_pf_suffix = "_pf" if PHYSICS_WEIGHT > 0 else ""
out_dir = runs_parent / (
    f"da_gps_chunked_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_mvagg_gine_metaaux_regce{_pf_suffix}{'_smoke' if SMOKE_TEST else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "voltage" if PHYSICS_WEIGHT > 0 else "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", "0.1",
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "96",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", "0",
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", "0.1",
    "--lambda_reg", "0.1",
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience", str(PATIENCE),
    "--seed", str(SMOKE_SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--log_every", "10",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
]

if PHYSICS_WEIGHT > 0:
    _pf_flags = [
        "--pf_data_root", str(PF_DATA_ROOT),
        "--loss_power_balance_weight", str(PHYSICS_WEIGHT),
        "--pf_sparse_y", "1",
        "--pf_huber_delta_kw", "10",
        "--pf_detach_controls",
    ]
    _bal = Path(PF_BALANCE_NODE_LIST_CSV)
    if not _bal.is_absolute():
        _bal = (REPO / _bal).resolve()
    if not _bal.is_file():
        raise FileNotFoundError(f"PF_BALANCE_NODE_LIST_CSV not found: {_bal}")
    _pf_flags.extend(["--pf_balance_node_list_csv", str(_bal)])
    cmd.extend(_pf_flags)

_mode = "physics-informed" if PHYSICS_WEIGHT > 0 else "baseline"
print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} ({_mode})")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_run_name = out_dir.name
print("\n=== Section 8 prerequisite ===")
print(f"Baseline run folder:  {out_dir.resolve()}")
print(f"Section 8 expects:    {MYDRIVE_DATA / 'runs' / _run_name}")
print("Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)")
if _on_colab() and not str(out_dir.resolve()).startswith(str(MYDRIVE_DATA.resolve())):
    print(
        "WARNING: run dir is NOT under MyDrive/datasets_gnn2 — "
        "copy to Drive before disconnecting the VM or Section 8 will fail."
    )
else:
    print("Checkpoints are on Drive; safe to run Section 8 after this session ends.")
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SMOKE_SEED if SMOKE_TEST else 42,
)



## Fine-tune shipped DA-GPS checkpoint on no-BESS + with-DER (compatibility mvagg)

This cell **continues training** from your current inference checkpoint:

`da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE` (**h=96, L=2, heads=2**, reg CE, 4 meta-aux heads).

It uses the **compatibility** `gnn_node_features_and_targets_mvagg.csv` from the with-BESS generator (BESS P/Q folded into `p_load_kw` / `q_load_kvar`), so the feature schema stays **`p_load_kw,q_load_kvar,p_pv_kw` + PE** — same as inference.

### Google Drive layout (upload once)

```
MyDrive/datasets_gnn2/
├── checkpoints/
│   └── da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/   ← upload whole folder from PC
│       ├── training_last.pt
│       ├── x_mean.pt  x_std.pt  y_mean.pt  y_std.pt
│       ├── pv_mean.pt  pv_std.pt
│       ├── reg_class_values.pt  reg_class_tables.json
│       └── (optional) da_gps_multitask_best.pt
├── original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/
│   └── run_*/gnn_node_features_and_targets_mvagg.csv  ...
├── original_8500_unbalanced_chunked_with_bess_aug_2000_40/   ← from cell 3 above
│   └── run_*/gnn_node_features_and_targets_mvagg.csv  ...
├── cache/          ← tensor caches (created automatically)
└── runs/           ← new fine-tune output folders
```

**PC source for the checkpoint folder:**

`GNN2/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/`

Zip that folder → upload to Drive → unzip under `MyDrive/datasets_gnn2/checkpoints/`.

Colab also needs the **GNN2 code repo** cloned to `/content/GNN2` (or set `GNN2_REPO_ROOT`).

Run **cell 3** (with-BESS dataset generation) first, or copy finished chunks to the Drive path above.

In [ ]:
# --- Fine-tune shipped DA-GPS checkpoint on blended no-BESS + with-DER mvagg chunks ---
import os
import sys
import subprocess
import datetime
import warnings
import fnmatch
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"

# ---------------- user toggles ----------------
BLEND_NO_BESS = False
WITHDER_ONLY = True
FINETUNE_SMOKE = False
EPOCHS = None

NOBESS_CHUNK_PARENT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
WITHDER_CHUNK_PARENT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_with_bess_aug_2000_40"
BLENDED_CHUNK_PARENT = MYDRIVE_DATA / "chunk_parents/blended_nobess_withder"

INIT_RUN_DIR = MYDRIVE_DATA / "checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
INIT_CHECKPOINT = INIT_RUN_DIR / "training_last.pt"

# Must match shipped inference checkpoint (folder name "l4" is legacy)
MODEL_HIDDEN = 96
MODEL_LAYERS = 2
MODEL_HEADS = 2
MODEL_NODE_EMB_DIM = 4
N_SYSTEM_TOKENS = 10
META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"

FINETUNE_EPOCHS = int(EPOCHS) if EPOCHS is not None else (15 if FINETUNE_SMOKE else 60)
FINETUNE_PATIENCE = 5 if FINETUNE_SMOKE else 15
FINETUNE_LR = 1e-4
FINETUNE_SEED = 20420231
REFRESH_BLEND_SYMLINKS = False    # True = rebuild blended chunk_parent symlinks

SMOKE_NOBESS_CHUNKS = 3
SMOKE_WITHDER_CHUNKS = 2


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_repo() -> Path:
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
    elif _on_colab() and (Path("/content/GNN2") / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
        p = Path("/content/GNN2").resolve()
    else:
        p = Path.cwd().resolve()
    if not (p / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
        raise FileNotFoundError(
            f"GNN2 repo not found at {p}. Clone to /content/GNN2 on Colab or set GNN2_REPO_ROOT."
        )
    return p


def _sorted_run_dirs(parent: Path) -> list[Path]:
    return sorted(
        (p for p in parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(f"No run_*/gnn_node_index_master.csv under {chunk_parent}")
    return hits[0]


def _chunk_glob_exact(names: list[str]) -> str:
    return ",".join(names)


def _ensure_blended_chunk_parent(
    blend_root: Path,
    *,
    nobess_parent: Path,
    withder_parent: Path,
    nobess_names: list[str] | None,
    withder_names: list[str] | None,
    refresh: bool = False,
) -> Path:
    blend_root.mkdir(parents=True, exist_ok=True)
    pairs: list[tuple[Path, Path]] = []
    for nm in nobess_names or []:
        pairs.append((nobess_parent / nm, blend_root / nm))
    for nm in withder_names or []:
        pairs.append((withder_parent / nm, blend_root / nm))

    for src, dst in pairs:
        if not src.is_dir():
            raise FileNotFoundError(f"Missing chunk folder: {src}")
        if dst.exists() or dst.is_symlink():
            if refresh:
                dst.unlink(missing_ok=True)
            else:
                continue
        dst.symlink_to(src.resolve(), target_is_directory=True)
    return blend_root


def _preflight_chunk(p: Path) -> None:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
        "gnn_node_index_master.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p}")


def _preflight_init_run(init_dir: Path, ckpt: Path) -> None:
    if not ckpt.is_file():
        raise FileNotFoundError(f"INIT_CHECKPOINT not found: {ckpt}")
    required = [
        "x_mean.pt",
        "x_std.pt",
        "y_mean.pt",
        "y_std.pt",
        "reg_class_values.pt",
        "reg_class_tables.json",
        "pv_mean.pt",
        "pv_std.pt",
    ]
    missing = [f for f in required if not (init_dir / f).is_file()]
    if missing:
        raise FileNotFoundError(
            f"INIT_RUN_DIR missing files {missing}. Upload the full checkpoint folder from PC:\n"
            "  gnn2_architecture_search/attention checkpoints/"
            "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/"
        )


if _on_colab() and not _drive_mounted():
    from google.colab import drive

    drive.mount("/content/drive")

REPO = _resolve_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

if not NOBESS_CHUNK_PARENT.is_dir():
    raise FileNotFoundError(f"NOBESS_CHUNK_PARENT not found: {NOBESS_CHUNK_PARENT}")
if not WITHDER_CHUNK_PARENT.is_dir():
    raise FileNotFoundError(f"WITHDER_CHUNK_PARENT not found: {WITHDER_CHUNK_PARENT}")

nobess_runs = _sorted_run_dirs(NOBESS_CHUNK_PARENT)
withder_runs = _sorted_run_dirs(WITHDER_CHUNK_PARENT)
if not withder_runs:
    raise RuntimeError(f"No run_* folders under {WITHDER_CHUNK_PARENT} — run cell 3 first.")

if WITHDER_ONLY:
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    selected_nobess: list[Path] = []
    chunk_parent = WITHDER_CHUNK_PARENT
    chunk_glob = _chunk_glob_exact([p.name for p in selected_withder])
elif BLEND_NO_BESS:
    selected_nobess = nobess_runs[:SMOKE_NOBESS_CHUNKS] if FINETUNE_SMOKE else nobess_runs
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    chunk_parent = _ensure_blended_chunk_parent(
        BLENDED_CHUNK_PARENT,
        nobess_parent=NOBESS_CHUNK_PARENT,
        withder_parent=WITHDER_CHUNK_PARENT,
        nobess_names=[p.name for p in selected_nobess],
        withder_names=[p.name for p in selected_withder],
        refresh=REFRESH_BLEND_SYMLINKS,
    )
    chunk_glob = _chunk_glob_exact([p.name for p in selected_nobess] + [p.name for p in selected_withder])
else:
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    selected_nobess = []
    chunk_parent = WITHDER_CHUNK_PARENT
    chunk_glob = _chunk_glob_exact([p.name for p in selected_withder])

_preflight_init_run(INIT_RUN_DIR, INIT_CHECKPOINT)
node_pe = _find_node_pe_csv(chunk_parent)

for ch in _sorted_run_dirs(chunk_parent):
    if ch.name in {s.strip() for s in chunk_glob.split(",")}:
        _preflight_chunk(ch)

cache_root = MYDRIVE_DATA / "cache/da_gps_finetune_nobess_withder_gine"
gnn_cache_root = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
runs_parent = MYDRIVE_DATA / "runs"
cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = runs_parent / (
    f"da_gps_finetune_withder_l{MODEL_LAYERS}_h{MODEL_HIDDEN}_regce"
    f"{'_smoke' if FINETUNE_SMOKE else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

print("=== Fine-tune preflight ===")
print(f"REPO:              {REPO}")
print(f"INIT_RUN_DIR:      {INIT_RUN_DIR}")
print(f"INIT_CHECKPOINT:   {INIT_CHECKPOINT}")
print(f"CHUNK_PARENT:      {chunk_parent}")
print(f"CHUNK_GLOB:        {chunk_glob}")
print(f"no-BESS chunks:    {len(selected_nobess)}")
print(f"with-DER chunks:   {len(selected_withder)}")
print(f"NODE_PE_CSV:       {node_pe}")
print(f"OUT_DIR:           {out_dir}")
holdout_cmd_extra: list[str] = []
if WITHDER_ONLY:
    holdout_chunks = nobess_runs[:SMOKE_NOBESS_CHUNKS] if FINETUNE_SMOKE else nobess_runs
    holdout_cmd_extra = [
        "--eval_holdout_chunk_parent",
        str(NOBESS_CHUNK_PARENT),
        "--eval_holdout_chunk_glob",
        _chunk_glob_exact([p.name for p in holdout_chunks]),
        "--eval_holdout_label",
        "nobess_holdout",
        "--eval_holdout_seed",
        str(FINETUNE_SEED),
    ]
    holdout_label = f"{len(holdout_chunks)} no-BESS chunk(s) -> nobess_holdout"
elif BLEND_NO_BESS:
    holdout_label = "skipped (no-BESS already in training blend)"
else:
    holdout_label = "none (with-DER only, no holdout parent)"

print(f"EPOCHS/LR:         {FINETUNE_EPOCHS} / {FINETUNE_LR}")
print(f"HOLDOUT:           {holdout_label}")
if WITHDER_ONLY:
    print(f"HOLDOUT_PARENT:    {NOBESS_CHUNK_PARENT}")
print("=======================\n")

cmd = [
    sys.executable,
    "-u",
    "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent",
    str(chunk_parent),
    "--chunk_subdir_glob",
    chunk_glob,
    "--nodes_csv",
    "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv",
    "gnn_edges_phase_static.csv",
    "--meta_csv",
    "gnn_sample_meta.csv",
    "--node_feature_cols",
    "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv",
    str(node_pe),
    "--node_pe_cols",
    "auto",
    "--n_system_tokens",
    str(N_SYSTEM_TOKENS),
    "--aux_meta_cols",
    META_AUX_COLS,
    "--lambda_pv",
    "0.1",
    "--out_dir",
    str(out_dir),
    "--cache_dir",
    str(cache_root),
    "--bootstrap_gnn_cache_dir",
    str(gnn_cache_root),
    "--init_checkpoint",
    str(INIT_CHECKPOINT),
    "--init_run_dir",
    str(INIT_RUN_DIR),
    "--drop_samples_unseen_reg_taps",
    "--eval_before_train",
    "--epochs",
    str(FINETUNE_EPOCHS),
    "--batch_size",
    "96",
    "--hidden",
    str(MODEL_HIDDEN),
    "--layers",
    str(MODEL_LAYERS),
    "--heads",
    str(MODEL_HEADS),
    "--node_emb_dim",
    str(MODEL_NODE_EMB_DIM),
    "--edge_emb_dim",
    "0",
    "--lr",
    str(FINETUNE_LR),
    "--weight_decay",
    "1e-5",
    "--lambda_cap",
    "0.1",
    "--lambda_reg",
    "0.1",
    "--reg_loss",
    "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience",
    str(FINETUNE_PATIENCE),
    "--seed",
    str(FINETUNE_SEED),
    "--train_frac",
    "0.80",
    "--val_frac",
    "0.10",
    "--sample_frac",
    "1.0",
    "--num_workers",
    "0" if os.name == "nt" else "4",
    "--log_every",
    "10",
    "--checkpoint_every",
    "10",
    "--early_stop_on",
    "total",
    "--dropout",
    "0.1",
] + holdout_cmd_extra

print("Running:\n ", " ".join(cmd), "\n", flush=True)
with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nFine-tune completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
print("\n=== Inference / warm-start ===")
print(f"RUN_DIR     = {out_dir.resolve()}")
print(f"CHECKPOINT  = {(out_dir / 'training_last.pt').resolve()}")
print("Point cell 9 bootstrap or daily-compare at the paths above after training.")